# Tracker Smoke Test

Verify that ByteTrack assigns persistent IDs across consecutive frames.
Diagnostic only: tests the integration before moving to calibration (Stage 2).

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "supervision"])


In [ ]:
import sys
sys.path.append("..")

import cv2
from pipeline.detection.tracker import Tracker

%load_ext autoreload
%autoreload 2

In [ ]:
tracker = Tracker("../pipeline/detection/football_yolo26n_best.pt", conf=0.1)
src = r"C:\Users\rohan\Desktop\Quant Sports Project\Tester video\08fd33_4.mp4"

cap = cv2.VideoCapture(src)
assert cap.isOpened(), f"Failed to open {src}"

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {fps} fps, {total_frames} frames")

In [ ]:
frame_count = 0
max_frames = 100
id_history = {}

while frame_count < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    
    tracks, _ = tracker.track_frame(frame)
    
    if tracks.tracker_id is not None:
        ids = sorted(set(int(id_) for id_ in tracks.tracker_id))
        id_history[frame_count] = ids
        
        if frame_count % 20 == 0:
            print(f"Frame {frame_count:3d}: {len(tracks)} objects, IDs: {ids}")
    else:
        print(f"Frame {frame_count:3d}: no tracks")
    
    frame_count += 1

cap.release()
print(f"\nProcessed {frame_count} frames")

In [ ]:
print("ID Persistence Check:")
print("=" * 50)

if id_history:
    for frame_num in sorted(id_history.keys())[:10]:
        ids = id_history[frame_num]
        print(f"Frame {frame_num}: {ids}")
    
    id_persistence = {}
    for frame_num, ids in id_history.items():
        for id_ in ids:
            id_persistence[id_] = id_persistence.get(id_, 0) + 1
    
    persistent_ids = {id_: count for id_, count in id_persistence.items() if count >= 2}
    print(f"\nTotal unique IDs: {len(id_persistence)}")
    print(f"IDs appearing 2+ frames: {len(persistent_ids)}")
    print(f"Persistence rate: {len(persistent_ids) / len(id_persistence) * 100:.1f}%")
else:
    print("No tracks found")

In [ ]:
import os
from pipeline.common.drawing import annotate_frame

cap = cv2.VideoCapture(src)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs("runs", exist_ok=True)
out = cv2.VideoWriter("runs/tracker_visual_check.avi", cv2.VideoWriter_fourcc(*"XVID"), fps, (w, h))

for i in range(50):
    ok, frame = cap.read()
    if not ok:
        break
    
    tracks, detection_raw = tracker.track_frame(frame)
    annotated = annotate_frame(frame, detection_raw)
    out.write(annotated)

cap.release()
out.release()
print("Saved to runs/tracker_visual_check.avi")

In [ ]:
import time
import numpy as np

cap = cv2.VideoCapture(src)
times = []

for i in range(50):
    ok, frame = cap.read()
    if not ok:
        break
    
    start = time.perf_counter()
    tracks, _ = tracker.track_frame(frame)
    elapsed = time.perf_counter() - start
    times.append(elapsed * 1000)

cap.release()

print(f"Mean: {np.mean(times):.2f}ms")
print(f"P95: {np.percentile(times, 95):.2f}ms")
print(f"Max: {np.max(times):.2f}ms")